# DDR Parquet Inspection Notebook

This notebook reproduces the exact exploratory analysis we ran in terminal:

1. Inventory all parquet files and inspect metadata
2. Identify likely join keys across cells and foci tables
3. Find candidate label columns in classified cells
4. Quantify overlap between cells and foci keys
5. Profile feature families in `MultiFoci_combined.parquet`

The notebook intentionally focuses only on dataset inspection and key validation.

In [1]:
from pathlib import Path
import re

import pandas as pd
import pyarrow.parquet as pq

# Data root
ROOT = Path('/nfs/roberts/pi/pi_sk2433/shared/JohnLock_2026_DDR')

# Core files used throughout inspection
CELLS_FILE = ROOT / 'cells_classified-003.parquet'
MULTI_FILE = ROOT / 'MultiFoci_combined.parquet'
NUCLEAR_FILE = ROOT / '260306_DDR23FOCINuclearFoci.csv.parquet'

print('ROOT exists:', ROOT.exists())
print('Cells file exists:', CELLS_FILE.exists())
print('Multi foci file exists:', MULTI_FILE.exists())
print('Nuclear foci file exists:', NUCLEAR_FILE.exists())

ROOT exists: True
Cells file exists: True
Multi foci file exists: True
Nuclear foci file exists: True


## 1) Inventory All Parquet Files (Metadata Only)

This uses parquet metadata only (no full dataframe load), which is safe for large files.

In [2]:
files = sorted(ROOT.glob('*.parquet'))
print('num parquet files:', len(files))

inventory_rows = []
for fp in files:
    pf = pq.ParquetFile(fp)
    schema_names = pf.schema.names
    row = {
        'file': fp.name,
        'rows': pf.metadata.num_rows,
        'cols': len(schema_names),
        'row_groups': pf.metadata.num_row_groups,
        'first_10_columns': schema_names[:10],
    }
    inventory_rows.append(row)

inventory_df = pd.DataFrame(inventory_rows).sort_values('rows', ascending=False)
inventory_df

num parquet files: 14


,file,rows,cols,row_groups,first_10_columns
3,260306_DDR23FOCIFMRP_Mask_Foci.csv.parquet,2194164,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
2,260306_DDR23FOCIFANCD2_Mask_Foci.csv.parquet,1078755,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
11,260306_DDR23FOCIph2ax_Mask_Foci.csv.parquet,620302,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
4,260306_DDR23FOCIHistone_Mask_Foci.csv.parquet,461184,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
6,260306_DDR23FOCIRAD51_Mask_Foci.csv.parquet,229677,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
8,260306_DDR23FOCIUbiq_Mask_Foci.csv.parquet,151298,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
5,260306_DDR23FOCINuclearFoci.csv.parquet,150195,2720,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
0,260306_DDR23FOCIBCL2_Mask_Foci.csv.parquet,146676,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
12,MultiFoci_combined.parquet,114484,2714,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."
10,260306_DDR23FOCIpATM_Mask_Foci.csv.parquet,101995,2712,1,"[ImageNumber, ObjectNumber, Metadata_Chan_ID, ..."


## 2) Key / Join Column Discovery in Core Tables

This scans three core files for likely keys (`ImageNumber`, `ObjectNumber`, `Parent_*`, metadata fields).

In [3]:
core_files = {
    'cells': CELLS_FILE,
    'multi': MULTI_FILE,
    'nuclear': NUCLEAR_FILE,
}

key_terms = [
    'ObjectNumber', 'ImageNumber', 'Parent', 'Children',
    'Metadata_Well_ID', 'Metadata_Field', 'Metadata_Marker',
    'Class', 'class', 'Label', 'label', 'Status', 'status', 'Prediction', 'prediction'
]

for tag, fp in core_files.items():
    pf = pq.ParquetFile(fp)
    cols = pf.schema.names
    hits = [c for c in cols if any(t in c for t in key_terms)]

    print(f'\n=== {tag}: {fp.name} ===')
    print('num_cols:', len(cols))
    print('matched_cols_count:', len(hits))
    for c in hits[:120]:
        print(c)
    if len(hits) > 120:
        print('...')

candidate_cols = [
    'ImageNumber', 'ObjectNumber', 'Metadata_Well_ID', 'Metadata_Field', 'Metadata_Marker',
    'Parent_Nuclei', 'Parent_Cells', 'Children_Nuclei_Count', 'Children_Cells_Count',
    'Nuclei_ObjectNumber', 'Nuclei_ImageNumber', 'Cytoplasm_ObjectNumber', 'Cytoplasm_ImageNumber',
    'Class', 'class', 'Label', 'label', 'Status', 'status', 'prediction', 'predicted_class',
    'cell_class', 'Cell_Class', 'Nuclei_Class',
]

for tag, fp in core_files.items():
    all_cols = pq.ParquetFile(fp).schema.names
    use_cols = [c for c in candidate_cols if c in all_cols]
    if not use_cols:
        print(f'\n[{tag}] no candidate columns found in tiny-read step')
        continue

    tiny = pd.read_parquet(fp, columns=use_cols).head(5)
    print(f'\n[{tag}] tiny read columns:', use_cols)
    display(tiny)
    for c in use_cols:
        vals = tiny[c].dropna().unique().tolist()
        print(f'  {c} sample uniques: {vals[:6]}')


=== cells: cells_classified-003.parquet ===
num_cols: 27926
matched_cols_count: 25
Cytoplasm_ImageNumber
Cytoplasm_ObjectNumber
Cytoplasm_Metadata_Field
Cytoplasm_Metadata_Marker
Cytoplasm_Metadata_Well_ID
Cytoplasm_Parent_Cell_Bodies
Cytoplasm_Parent_Nuclei_NonEdge
Parent_CellBody_Identity
Parent_Nuclei_Identity
CellBody_ImageNumber
CellBody_ObjectNumber
CellBody_Metadata_Field
CellBody_Metadata_Marker
CellBody_Metadata_Well_ID
CellBody_Children_Cytoplasm_Count
CellBody_Children_Nuclei_NonEdge_Count
CellBody_Neighbors_FirstClosestObjectNumber_Adjacent
CellBody_Neighbors_SecondClosestObjectNumber_Adjacent
CellBody_Parent_Cell_Bodies_NonEdge
Nuclei_ImageNumber
Nuclei_ObjectNumber
Nuclei_Metadata_Field
Nuclei_Metadata_Marker
Nuclei_Parent_Nuclei_NonEdge
Parent_ID

=== multi: MultiFoci_combined.parquet ===
num_cols: 2714
matched_cols_count: 15
ImageNumber
ObjectNumber
Metadata_Field
Metadata_Marker
Metadata_Well_ID
Neighbors_FirstClosestObjectNumber_Nuclei_NonEdge_Expanded
Neighbors_Seco

,Nuclei_ObjectNumber,Nuclei_ImageNumber,Cytoplasm_ObjectNumber,Cytoplasm_ImageNumber
0,1.0,1.0,1,1
1,3.0,1.0,2,1
2,2.0,1.0,3,1
3,4.0,1.0,4,1
4,6.0,1.0,5,1


  Nuclei_ObjectNumber sample uniques: [1.0, 3.0, 2.0, 4.0, 6.0]
  Nuclei_ImageNumber sample uniques: [1.0]
  Cytoplasm_ObjectNumber sample uniques: [1, 2, 3, 4, 5]
  Cytoplasm_ImageNumber sample uniques: [1]

[multi] tiny read columns: ['ImageNumber', 'ObjectNumber', 'Metadata_Well_ID', 'Metadata_Field', 'Metadata_Marker']


,ImageNumber,ObjectNumber,Metadata_Well_ID,Metadata_Field,Metadata_Marker
0,1,1,B02,0,NaN
1,1,2,B02,0,NaN
2,1,3,B02,0,NaN
3,1,4,B02,0,NaN
4,1,5,B02,0,NaN


  ImageNumber sample uniques: [1]
  ObjectNumber sample uniques: [1, 2, 3, 4, 5]
  Metadata_Well_ID sample uniques: ['B02']
  Metadata_Field sample uniques: [0]
  Metadata_Marker sample uniques: []

[nuclear] tiny read columns: ['ImageNumber', 'ObjectNumber', 'Metadata_Well_ID', 'Metadata_Field', 'Metadata_Marker']


,ImageNumber,ObjectNumber,Metadata_Well_ID,Metadata_Field,Metadata_Marker
0,1,1,B02,0,NaN
1,1,2,B02,0,NaN
2,1,3,B02,0,NaN
3,1,4,B02,0,NaN
4,1,5,B02,0,NaN


  ImageNumber sample uniques: [1]
  ObjectNumber sample uniques: [1, 2, 3, 4, 5]
  Metadata_Well_ID sample uniques: ['B02']
  Metadata_Field sample uniques: [0]
  Metadata_Marker sample uniques: []


## 3) Candidate Label Discovery in Classified Cells

This searches for label-like columns (`class`, `state`, `status`, `pred`, etc.) and reports class balance.

In [4]:
cells_cols = pq.ParquetFile(CELLS_FILE).schema.names
label_pattern = re.compile(r'(class|label|status|state|pred|cluster|phenotype|type)', re.IGNORECASE)
label_like = [c for c in cells_cols if label_pattern.search(c)]

print('label-like columns count:', len(label_like))
for c in label_like:
    print(c)

if label_like:
    labels_df = pd.read_parquet(CELLS_FILE, columns=label_like)
    print('\n=== label candidate summary (nunique <= 30) ===')
    for c in label_like:
        s = labels_df[c]
        nun = s.nunique(dropna=True)
        if nun <= 30:
            print(f'\n{c} | dtype={s.dtype} | nunique={nun}')
            print(s.value_counts(dropna=False).head(15).to_string())

label-like columns count: 3
CC_cluster
Cell_State_Thr
Cell_State_Score

=== label candidate summary (nunique <= 30) ===

CC_cluster | dtype=int32 | nunique=5
CC_cluster
1    12571
3     6729
4     3682
2     2961
0     1096

Cell_State_Thr | dtype=object | nunique=6
Cell_State_Thr
Intermediate            23397
Resisting Cell Death     1498
Stressed                  725
Proliferative             703
Early Apoptotic           439
Late Apoptotic            277

Cell_State_Score | dtype=object | nunique=6
Cell_State_Score
Intermediate            19017
Proliferative            2364
Resisting Cell Death     1950
Late Apoptotic           1596
Stressed                 1365
Early Apoptotic           747


## 4) Key Overlap Between Classified Cells and Foci Tables

This validates whether supervised joins are feasible using nucleus/object keys.

In [5]:
cells = pd.read_parquet(
    CELLS_FILE,
    columns=['Nuclei_ImageNumber', 'Nuclei_ObjectNumber', 'CC_cluster', 'Cell_State_Thr', 'Cell_State_Score']
).rename(columns={'Nuclei_ImageNumber': 'ImageNumber', 'Nuclei_ObjectNumber': 'ObjectNumber'})

multi = pd.read_parquet(MULTI_FILE, columns=['ImageNumber', 'ObjectNumber', 'Metadata_Well_ID', 'Metadata_Field'])
nuclear = pd.read_parquet(NUCLEAR_FILE, columns=['ImageNumber', 'ObjectNumber', 'Parent_Nuclei_NonEdge', 'Parent_ID'])

cells_keys = cells[['ImageNumber', 'ObjectNumber']].dropna().drop_duplicates()
multi_keys = multi[['ImageNumber', 'ObjectNumber']].dropna().drop_duplicates()
nuclear_keys = nuclear[['ImageNumber', 'ObjectNumber']].dropna().drop_duplicates()

cells_multi = cells_keys.merge(multi_keys, on=['ImageNumber', 'ObjectNumber'], how='inner')
cells_nuclear = cells_keys.merge(nuclear_keys, on=['ImageNumber', 'ObjectNumber'], how='inner')

print('cells unique nuclei keys:', len(cells_keys))
print('multi unique object keys:', len(multi_keys))
print('nuclear unique object keys:', len(nuclear_keys))
print('cells∩multi:', len(cells_multi), f"({100 * len(cells_multi) / len(cells_keys):.2f}% of cells)")
print('cells∩nuclear:', len(cells_nuclear), f"({100 * len(cells_nuclear) / len(cells_keys):.2f}% of cells)")

mf_counts = multi.groupby(['ImageNumber', 'ObjectNumber']).size().rename('multi_foci_rows').reset_index()
join_counts = cells_keys.merge(mf_counts, on=['ImageNumber', 'ObjectNumber'], how='left')
print('cells with >=1 multi foci rows:', int((join_counts['multi_foci_rows'].fillna(0) > 0).sum()))
print('median multi_foci_rows among matched:', float(join_counts['multi_foci_rows'].dropna().median()))
print('p95 multi_foci_rows among matched:', float(join_counts['multi_foci_rows'].dropna().quantile(0.95)))

matched_cells = cells.merge(multi_keys, on=['ImageNumber', 'ObjectNumber'], how='inner')
for col in ['CC_cluster', 'Cell_State_Thr', 'Cell_State_Score']:
    print(f'\n{col} matched distribution:')
    print(matched_cells[col].value_counts(dropna=False).to_string())

cells unique nuclei keys: 27039
multi unique object keys: 77819
nuclear unique object keys: 150195
cells∩multi: 21603 (79.90% of cells)
cells∩nuclear: 23141 (85.58% of cells)
cells with >=1 multi foci rows: 21603
median multi_foci_rows among matched: 2.0
p95 multi_foci_rows among matched: 4.0

CC_cluster matched distribution:
CC_cluster
1    8719
3    6185
4    3037
2    2768
0     894

Cell_State_Thr matched distribution:
Cell_State_Thr
Intermediate            18329
Resisting Cell Death     1467
Proliferative             662
Stressed                  653
Early Apoptotic           251
Late Apoptotic            241

Cell_State_Score matched distribution:
Cell_State_Score
Intermediate            14622
Proliferative            2240
Resisting Cell Death     1840
Late Apoptotic           1283
Stressed                 1221
Early Apoptotic           397


## 5) MultiFoci Feature Family Profiling

This scans `MultiFoci_combined.parquet` columns for spatial, intensity, shape, and `Parent_*` feature families.

In [6]:
multi_cols = pq.ParquetFile(MULTI_FILE).schema.names

patterns = {
    'spatial': re.compile(r'(center|centroid|location|x|y|z|radial|distance)', re.IGNORECASE),
    'intensity': re.compile(r'(intensity|mean|median|max|min|integrated|sum)', re.IGNORECASE),
    'shape': re.compile(r'(area|perimeter|eccentric|compact|solidity|extent|major|minor|formfactor|feret)', re.IGNORECASE),
    'parent_marker': re.compile(r'^Parent_', re.IGNORECASE),
}

for name, pat in patterns.items():
    hits = [c for c in multi_cols if pat.search(c)]
    print(f'\n[{name}] count={len(hits)}')
    for c in hits[:80]:
        print(c)
    if len(hits) > 80:
        print('...')

sample_cols = [
    'ImageNumber', 'ObjectNumber', 'Metadata_Well_ID', 'Metadata_Field',
    'Parent_ID', 'Parent_Type',
    'Location_Center_X', 'Location_Center_Y',
    'AreaShape_Area', 'AreaShape_Perimeter', 'AreaShape_Eccentricity',
    'Parent_pATM foci', 'Parent_a53P1 foci', 'Parent_RAD51 foci', 'Parent_FANCD2 foci', 'Parent_RPA70 foci'
]
use_cols = [c for c in sample_cols if c in multi_cols]
print('\ncolumns used for tiny sample:', use_cols)

if use_cols:
    tiny_multi = pd.read_parquet(MULTI_FILE, columns=use_cols).head(8)
    display(tiny_multi)


[spatial] count=2676
Metadata_Cycle_ID
Metadata_ExpDate
Metadata_FileLocation
FileName_Cycle01_DAPI
FileName_Cycle01_DIC
FileName_Cycle01_Ku7080
FileName_Cycle01_RAD51
FileName_Cycle01_pH2AX
FileName_Cycle02_CDK2
FileName_Cycle02_DAPI
FileName_Cycle02_DIC
FileName_Cycle02_IRF3
FileName_Cycle02_pATM(1981)
FileName_Cycle03_ATM
FileName_Cycle03_Cleaved PARP1
FileName_Cycle03_DAPI
FileName_Cycle03_DIC
FileName_Cycle03_EXO1
FileName_Cycle04_BRCA1
FileName_Cycle04_DAPI
FileName_Cycle04_DIC
FileName_Cycle04_P53
FileName_Cycle04_pP53
FileName_Cycle05_BRCA2
FileName_Cycle05_DAPI
FileName_Cycle05_DIC
FileName_Cycle05_XPD
FileName_Cycle05_cGAS
FileName_Cycle06_DAPI
FileName_Cycle06_DIC
FileName_Cycle06_ERCC1
FileName_Cycle06_MRE11A
FileName_Cycle06_SSB
FileName_Cycle07_CHEK1
FileName_Cycle07_CHEK2
FileName_Cycle07_DAPI
FileName_Cycle07_DIC
FileName_Cycle07_FANCD2
FileName_Cycle08_53BP1
FileName_Cycle08_Cyclin E1
FileName_Cycle08_DAPI
FileName_Cycle08_DIC
FileName_Cycle08_Fibrillarin
FileName_Cyc

,ImageNumber,ObjectNumber,Metadata_Well_ID,Metadata_Field,Parent_ID,Parent_Type,Location_Center_X,Location_Center_Y,AreaShape_Area,AreaShape_Perimeter,AreaShape_Eccentricity,Parent_pATM foci,Parent_a53P1 foci,Parent_RAD51 foci,Parent_FANCD2 foci,Parent_RPA70 foci
0,1,1,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,344.875000,348.875000,8,0.213831,0.613654,1.0,NaN,NaN,NaN,NaN
1,1,2,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,704.714294,389.857147,7,0.167184,0.724963,2.0,NaN,NaN,NaN,NaN
2,1,3,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,706.818176,390.909088,11,0.313208,0.603224,3.0,NaN,NaN,NaN,NaN
3,1,4,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,547.500000,398.500000,12,0.299011,0.000000,4.0,NaN,NaN,NaN,NaN
4,1,5,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,363.000000,401.500000,6,0.159123,0.791463,5.0,NaN,NaN,NaN,NaN
5,1,6,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,567.411743,411.882355,17,0.394922,0.627208,6.0,NaN,NaN,NaN,NaN
6,1,7,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,558.125000,438.875000,8,0.213831,0.613654,7.0,NaN,NaN,NaN,NaN
7,1,8,B02,0,251212_DDR23_B02_0000_Cycle01_Chan01_ex405nm_D...,Nuclei,537.363647,440.909088,11,0.289886,0.709559,8.0,NaN,NaN,NaN,NaN


## 6) Optional Next Check (if you want)

A good next notebook section would test whether marker-specific foci files can be stacked cleanly and joined to the same nucleus keys without introducing duplicate parent mappings.